In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Input, InputLayer

# Load energy data
clean_csv_path = os.path.join(os.path.dirname(os.getcwd()), 'clean_energy_data.csv')
energy_df = pd.read_csv(clean_csv_path, parse_dates=['utc_timestamp'])

# Feature columns
# Removed Feature Columns "shortwave_radiation", "wind_speed" for future analyses
meteorological_features = ["temp", "humidity", "cloudcover",  "wind_direction", "pressure"]
temporal_features = ['hour', 'day', 'month']
all_features = meteorological_features + temporal_features

# Output directory for SHAP plots
output_dir = os.path.join(os.getcwd(), "removed_feature_shap_outputs")
os.makedirs(output_dir, exist_ok=True)

# Summary collector
summary_records = []


# === Main modeling function ===
def model_pipeline(df, target_label, task_name, country_code):
    X = df[all_features]
    y = df[target_label]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # === FFN Model ===
    ffn = Sequential([
        Input(shape=(X_train_scaled.shape[1],)),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    ffn.compile(optimizer='adam', loss='mse')
    ffn.fit(X_train_scaled, y_train, epochs=100, batch_size=32, verbose=0)

    # === GRU Model ===
    X_train_gru = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_gru = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

    gru = Sequential([
        InputLayer((1, X_train_scaled.shape[1])),
        GRU(64),
        Dense(1)
    ])
    gru.compile(optimizer='adam', loss='mse')
    gru.fit(X_train_gru, y_train, epochs=100, batch_size=32, verbose=0)

    # === Evaluation ===
    ffn_preds = ffn.predict(X_test_scaled, verbose=0).flatten()
    gru_preds = gru.predict(X_test_gru, verbose=0).flatten()

    print(f"\n--- {task_name.upper()} ({country_code}) ---")
    print(f"FFN  → RMSE: {np.sqrt(mean_squared_error(y_test, ffn_preds)):.2f} | R²: {r2_score(y_test, ffn_preds):.2f}")
    print(f"GRU  → RMSE: {np.sqrt(mean_squared_error(y_test, gru_preds)):.2f} | R²: {r2_score(y_test, gru_preds):.2f}")

    # === SHAP FFN ===
    def ffn_predict(x):
        return ffn.predict(x, verbose=0).reshape(-1)

    explainer_ffn = shap.KernelExplainer(ffn_predict, X_train_scaled)
    shap_values_ffn = explainer_ffn.shap_values(X_test_scaled)
    shap_matrix_ffn = np.array(shap_values_ffn)
    mean_abs_ffn = np.abs(shap_matrix_ffn).mean(axis=0)
    norm_ffn = mean_abs_ffn / mean_abs_ffn.max()

    # === SHAP GRU ===
    def gru_predict(x):
        return gru.predict(x.reshape((x.shape[0], 1, x.shape[1])), verbose=0).reshape(-1)

    explainer_gru = shap.KernelExplainer(gru_predict, X_train_scaled[:1000])
    shap_values_gru = explainer_gru.shap_values(X_test_scaled[:300])
    shap_matrix_gru = np.array(shap_values_gru)
    mean_abs_gru = np.abs(shap_matrix_gru).mean(axis=0)
    norm_gru = mean_abs_gru / mean_abs_gru.max()

    # === Save SHAP Plots + Collect Summary ===
    model_norm_pairs = [("FFN", norm_ffn), ("GRU", norm_gru)]
    for model_name, norm_array in model_norm_pairs:
        sorted_idx = np.argsort(norm_array)
        plt.figure(figsize=(6, 4))
        plt.barh(np.array(all_features)[sorted_idx], norm_array[sorted_idx], color="steelblue")
        plt.xlabel("Relative SHAP Importance (0-1)")
        plt.title(f"{task_name.upper()} - {country_code} - {model_name}")
        plt.tight_layout()
        filename = f"{country_code}_{task_name.replace(' ', '_')}_{model_name}_shap.png"
        plt.savefig(os.path.join(output_dir, filename))
        plt.close()

        for feature, score in zip(all_features, norm_array):
            summary_records.append({
                "Country": country_code,
                "Task": task_name,
                "Model": model_name,
                "Feature": feature,
                "NormalizedImportance": round(score, 4)
            })


# === Per-country data wrapper ===
def run_country_pipeline(country_code):
    weather_csv_path = os.path.join(os.path.dirname(os.getcwd()), f'weather_data/{country_code}_hourly_2015-2019.csv')
    if not os.path.exists(weather_csv_path):
        return

    weather_df = pd.read_csv(weather_csv_path, parse_dates=['timestamp'])

    solar_col = f"{country_code}_solar_generation_actual"
    wind_col = f"{country_code}_wind_onshore_generation_actual"
    if solar_col not in energy_df.columns or wind_col not in energy_df.columns:
        return

    merged_df = pd.merge(
        energy_df[['utc_timestamp', solar_col, wind_col]].copy(),
        weather_df,
        left_on=energy_df['utc_timestamp'].dt.tz_convert(None),
        right_on='timestamp',
        how='inner'
    )

    merged_df.dropna(inplace=True)
    merged_df['hour'] = merged_df['timestamp'].dt.hour
    merged_df['day'] = merged_df['timestamp'].dt.dayofweek
    merged_df['month'] = merged_df['timestamp'].dt.month

    model_pipeline(merged_df, solar_col, "solar", country_code)
    model_pipeline(merged_df, wind_col, "wind", country_code)


# === Run for all countries ===
weather_folder = os.path.join(os.path.dirname(os.getcwd()), 'weather_data')
weather_files = glob.glob(os.path.join(weather_folder, '*_hourly_2015-2019.csv'))

for path in weather_files:
    country_code = os.path.basename(path).split('_')[0]
    run_country_pipeline(country_code)

# === Save combined CSV ===
summary_df = pd.DataFrame(summary_records)
summary_csv_path = os.path.join(output_dir, "shap_feature_importance_summary.csv")
summary_df.to_csv(summary_csv_path, index=False)

# Display the DataFrame in notebook
print("\nSHAP summary saved to:", summary_csv_path)
print(summary_df.head(10))